# Retrieval Layer — Stage 1: Chunking the Corpus

Builds the retrieval half of a RAG system over the SDE methodology decision record.

**Why this exists.** The existing valuation pipeline formats the entire practice dataset
into one string and sends all of it on every question. That is correct for numbers — you
want the *exact* revenue figure, not the four rows most similar to the phrase "revenue."
But it left a gap: when the model explained *why* an expense qualified as an add-back, it
had nothing to draw on. Those rules lived in regex patterns and hardcoded constants, where
the model could not see them.

This notebook makes the rules retrievable. The pipeline stays split:

| Question | Path | Why |
|---|---|---|
| "What was 2023 revenue?" | existing deterministic lookup | exactness matters; retrieval would be a downgrade |
| "Why is automobile only 25%?" | retrieval (this notebook) | the answer is a passage, not a cell |

Kept separate from the main pipeline because retrieval is independently testable — it
needs no API key, no CSVs, no practice data. Only the markdown file.

## 1. Chunking

Retrieval searches **chunks**, not documents. Each chunk becomes one vector, and a chunk
covering four topics produces a vector that is a blurred average of all four — close to
nothing in particular, so it is retrieved for none of them. Too small fails the other way:
"$9,114.00 in FY2022" is precise and meaningless without the sentence defining it.

The target is a passage that answers one question completely.

**Strategies and tradeoffs:**

| Approach | Use when |
|---|---|
| Fixed size (every N chars) | text has no structure; accepts mid-sentence cuts |
| Paragraph | structure exists but headings don't |
| **Section / heading** | the document was written with meaningful headings |
| Semantic (detect topic shift by embedding similarity) | structure is unreliable and accuracy justifies the cost |

Section-based here, because the decision record was drafted so each section answers one
question. That structure *is* the chunking; the code only has to read it.

**Breadcrumbs.** `## 5. Known methodology issues` is a bare heading — its body is entirely
subsections. Chunked naively, its children survive but lose their parent, producing a
chunk labeled only "FY2022" with nothing in it saying *owner compensation*.

So each chunk carries its parent heading, written **into the chunk text** rather than
stored beside it. Metadata kept in a separate field is not embedded and cannot affect
whether a chunk is found. This generalizes: anything that should influence retrieval must
be inside the embedded text.

In [ ]:
## Cell 1 — parse the document into chunks

import re
from pathlib import Path

DOC = Path("SDE_Method_Decision_Record.md")

# encoding="utf-8" is not optional. Without it Python falls back to the platform
# default (cp1252 on Windows) and every em dash arrives as "â€”" — which then gets
# embedded, degrading the vectors silently.
text = DOC.read_text(encoding="utf-8")

# Split BEFORE each level-2 or level-3 heading.
#   \n(?=...)  lookahead: consume the newline, leave the "##" attached to its section
#   #{2,3}     "##" or "###" only — the "#" title line is not a section
chunks, h2 = [], None

for part in re.split(r'\n(?=#{2,3} )', text):
    part = part.strip()
    if not part.startswith('#'):
        continue                                  # drops the title block

    line    = part.split('\n', 1)[0]
    level   = len(line) - len(line.lstrip('#'))
    heading = line.lstrip('# ').strip()

    if level == 2:
        h2, breadcrumb = heading, heading
    else:
        breadcrumb = f"{h2} > {heading}" if h2 else heading

    body = part.split('\n', 1)[1].strip() if '\n' in part else ''
    if not body:
        continue                                  # bare heading: survives in its children's breadcrumbs

    chunks.append({"id": f"sde-{len(chunks):02d}",
                   "heading": breadcrumb,
                   "text": f"{breadcrumb}\n\n{body}"})

print(f"{len(chunks)} chunks\n")
for c in chunks:
    flag = "  <-- oversized" if len(c["text"]) > 2000 else ""
    print(f"  {c['id']}  {len(c['text']):>5}  {c['heading'][:65]}{flag}")

## 2. Splitting oversized sections

The owner-compensation section runs past 2,000 characters because it contains both the
reasoning (why employer payroll tax does not belong in the numerator) and the resulting
figures. Those answer different questions, so fusing them means every query retrieves both
and half of what reaches the model is irrelevant.

Splitting at a fixed character count would cut mid-table, so segments break only at
paragraph boundaries.

**Two thresholds, not one.** 2,000 triggers a split; 1,400 sizes the results. If both were
2,000, a 2,100-character section would yield one full segment and a useless 100-character
runt.

Sizes are in characters. Embedding models cap on *tokens* and silently truncate past the
limit — worse than an error, since nothing reports it. At roughly 4 characters per token
for English, 1,400 characters is about 350 tokens, well under any model's ceiling, so
retrieval quality binds before truncation does.

In [ ]:
## Cell 2 — split oversized chunks at paragraph boundaries

MAX_SEGMENT = 1400     # target size for a split segment
OVERSIZED   = 2000     # threshold above which a chunk is split at all

def pack(paragraphs, limit):
    """Group whole paragraphs into segments under `limit` characters.

    A single paragraph longer than `limit` becomes its own oversized segment rather
    than being cut — a slightly-too-long coherent passage beats a severed one.
    """
    segments, cur = [], []
    for p in paragraphs:
        candidate = cur + [p]
        # + 2*len(cur) accounts for the "\n\n" rejoining each pair
        if cur and sum(len(x) for x in candidate) + 2 * len(cur) > limit:
            segments.append(cur)
            cur = [p]
        else:
            cur = candidate
    if cur:
        segments.append(cur)
    return segments


final = []
for c in chunks:
    if len(c["text"]) <= OVERSIZED:
        final.append(c)
        continue

    body = c["text"].split("\n\n", 1)[1]          # drop breadcrumb before splitting
    for i, seg in enumerate(pack(body.split("\n\n"), MAX_SEGMENT)):
        label = c["heading"] if i == 0 else f"{c['heading']} (cont. {i+1})"
        final.append({"id": None, "heading": label,
                      "text": f"{label}\n\n" + "\n\n".join(seg)})

for i, c in enumerate(final):                      # ids must stay contiguous after splitting
    c["id"] = f"sde-{i:02d}"

chunks = final
print(f"{len(chunks)} chunks, max {max(len(c['text']) for c in chunks)} chars")

## 3. Verify and save

Sizes cannot tell you whether a chunk is about one thing — that requires reading the
boundaries. The first segment of a split section should hold the rule; the continuation
should hold the figures. A cut landing mid-table is invisible until retrieval quietly
returns the wrong half.

Saving to JSON freezes the chunk set. If chunking drifts between runs, retrieval quality
drifts with it and you cannot tell whether a change came from a new embedding model or
from different chunks.

In [ ]:
## Cell 3 — inspect boundaries, check invariants, persist

import json

for c in chunks:
    if "(cont." in c["heading"] or "5.2" in c["heading"]:
        print(f"--- {c['id']} · {c['heading'][:70]}")
        preview = c["text"].split("\n\n", 1)[1][:260]
        print("    " + preview.rsplit(" ", 1)[0].replace("\n", "\n    ") + " ...\n")

assert all(len(c["text"]) <= 2000 for c in chunks),      "a chunk is still oversized"
assert len({c["id"] for c in chunks}) == len(chunks),    "duplicate chunk ids"
assert all(c["heading"] in c["text"] for c in chunks),   "breadcrumb missing from text"

OUT = Path("sde_chunks.json")
OUT.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"OK — {len(chunks)} chunks, {sum(len(c['text']) for c in chunks):,} chars -> {OUT}")

## Next: Stage 2 — embedding

Retrieval needs vectors, so each chunk gets converted into a list of numbers positioning
it in a space where nearby means similar in meaning. The decision waiting there is which
model produces them — the plan is to run one locally, since embeddings are cheap to
compute and the corpus names a practice and its owners' compensation.